In [ ]:
# Repository bootstrap for relocated notebooks
from pathlib import Path
import os
import sys

def _find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "README.md").exists() and (candidate / "src").exists():
            return candidate
    return start

REPO_ROOT = _find_repo_root(Path.cwd().resolve())
os.chdir(REPO_ROOT)
for _extra_path in (REPO_ROOT, REPO_ROOT / "src"):
    _extra_str = str(_extra_path)
    if _extra_str not in sys.path:
        sys.path.insert(0, _extra_str)

DATA_DIR = REPO_ROOT / "data"
MODELS_DIR = REPO_ROOT / "models"
EVAL_OUTPUT_DIR = REPO_ROOT / "eval_output"
CONFIGS_DIR = REPO_ROOT / "configs"

# Runtime directory defaults for organized local artifacts
for _runtime_dir in (
    DATA_DIR / "household" / "raw",
    DATA_DIR / "household" / "splits",
    DATA_DIR / "household" / "logs",
    MODELS_DIR / "household" / "sb3",
    MODELS_DIR / "household" / "dt",
    MODELS_DIR / "aemo_sb3",
    MODELS_DIR / "aemo" / "dt",
):
    _runtime_dir.mkdir(parents=True, exist_ok=True)


In [ ]:
# Imports and simple env
import gymnasium as gym
import numpy as np
import matplotlib.pyplot as plt

# Use your real env or a toy one
from EnergySimEnv import SolarBatteryEnv

# A single‐env factory for vec-env
def make_env(df):
    return lambda: SolarBatteryEnv(df)

In [ ]:
# Build a DummyVecEnv
from stable_baselines3.common.vec_env import DummyVecEnv

# Suppose you have a list `training_dataset` of Polars‐DFs
# that you already transformed with helper.transform_polars_df
env_fns = [make_env(df) for df in training_dataset]
vec_env = DummyVecEnv(env_fns)
print("VecEnv has", vec_env.num_envs, "envs.")

In [ ]:
# Instantiate and train PPO
from stable_baselines3 import PPO

model = PPO(
    policy="MlpPolicy",
    env=vec_env,
    verbose=1,
    learning_rate=1e-4,
    gamma=0.99,
    n_steps=2048,
    batch_size=64,
)

# Train for 50k timesteps
model.learn(total_timesteps=50_000)

In [ ]:
# Evaluate
from stable_baselines3.common.evaluation import evaluate_policy

mean_reward, std_reward = evaluate_policy(
    model, 
    vec_env, 
    n_eval_episodes=5, 
    deterministic=True
)
print(f"Mean reward: {mean_reward:.2f} ± {std_reward:.2f}")

In [ ]:
# Plot learning curve (if available)
# If you passed a [Monitor](http://_vscodecontentref_/0) wrapper, you could load log and plt here.
# For demo, we just plot the final bar:
plt.bar(["PPO"], [mean_reward], yerr=[std_reward], capsize=5)
plt.ylabel("Reward"); plt.title("Evaluation")
plt.show()

In [ ]:
# Save & reload
model.save("ppo_solar_batt")
del model

# reload
model = PPO.load("ppo_solar_batt", env=vec_env)
print("Model reloaded and ready to run!")

In [ ]:
# Rollout demo
obs, _ = vec_env.reset()
for _ in range(24):
    action, _ = model.predict(obs, deterministic=True)
    obs, reward, terminated, truncated, info = vec_env.step(action)
    print(f"Step reward {reward}")
    if terminated.any() or truncated.any():
        break